# Speculative decoding: English, Hindi and Urdu

Compare target-only and assisted generation with Qwen. Record throughput, target forward calls, exact output matches and script scores. Results describe the tested model pair and questions; saved numerical mismatches and quality limitations must be reported.


**Data review:** see `DATA_REVIEW.md`. Historical final-run executions are archived in `results_rp105/benchmark_executed.ipynb`. This corrected notebook has not been run on the GPU.


## 1. Imports
- `torch`: runs models on the GPU
- `transformers`: loads Qwen + has speculative decoding built in
- `random`: shuffles the run order (see section 11)
- the rest: timing, maths, saving files

In [ ]:
import csv, json, random, time
import statistics as st
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

## 2. Settings

The measured historical sweep used K=2,4,6,8 and two repeats. New defaults also test K=1 and use three repeats. Set a new OUTNAME for each run. A fixed shuffle seed reproduces order, not hardware timing. Start the notebook in this project folder, or explicitly set HERE to its path.


In [ ]:
TARGET  = "Qwen/Qwen2.5-3B-Instruct"
DRAFT   = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW = 128
LANGS   = ["en", "hi", "ur"]
LIMIT   = None           # None = all 10 questions (~35 min). Set to 2 for a ~4 min trial run.
K_SWEEP = [1, 2, 4, 6, 8]   # HF's unpinned default behaves like K=20 (see section 6)
REPEATS = 3              # trials per measurement; median is reported
SEED    = 0              # fixes the shuffled run order, so the run is reproducible
REP_PENALTY = 1.05       # see section 7. 1.05 = Qwen's own setting; 1.0 = true argmax
OUTNAME = "results_new"   # folder for this run's output

# Works whether the kernel started in this folder or not.
HERE = Path.cwd() if (Path.cwd() / "prompts.json").exists() else Path(globals().get("__vsc_ipynb_file__", ".")).parent
assert (HERE / "prompts.json").exists(), f"prompts.json not found next to the notebook (looked in {HERE})"
OUT = HERE / OUTNAME
assert not (OUT / "raw.csv").exists(), "Choose a new OUTNAME; measurements already exist."
OUT.mkdir(parents=True, exist_ok=True)
import sys
sys.path.insert(0, str(HERE.resolve()))

LANG_NAMES = {"en": "English", "hi": "Hindi", "ur": "Urdu"}
SYSTEM = "Answer in the same language as the question."
print("working in", HERE)

## 3. Script screen

Count the fraction of alphabetic Unicode code points in the expected script. A 70% threshold screens script switching; it does not establish fluency, factuality or even language identity. New code checks each actual output, including divergent speculative answers.


In [ ]:
SCRIPTS = {
    "en": [(0x0041, 0x005A), (0x0061, 0x007A)],
    "hi": [(0x0900, 0x097F)],
    "ur": [(0x0600, 0x06FF), (0x0750, 0x077F), (0xFB50, 0xFDFF), (0xFE70, 0xFEFF)],
}
LANG_OK_THRESHOLD = 0.7

def script_ratio(text, lang):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    hit = sum(any(a <= ord(c) <= b for a, b in SCRIPTS[lang]) for c in letters)
    return hit / len(letters)

# quick sanity test: should print 1.0, 1.0, 0.0
print(script_ratio("Rain forms when", "en"),
      script_ratio("बारिश तब बनती है", "hi"),
      script_ratio("Rain forms when", "ur"))

## 4. Counting big-model passes
- A "hook" is a tiny function that runs every time the big model runs
- Each run = one expensive pass, so we count them
- **tau = tokens generated ÷ passes**
  - Normal decoding: tau = 1 (one token per pass)
  - Speculative: tau > 1 means the helper's guesses got accepted
- tau is the clearest proof the trick is working

**Also here: a GPU clock probe.** This exists because of a real failure. Partway through an
earlier sweep the laptop's ASUS power profile changed and the GPU silently dropped from
**2580 MHz to 930 MHz**. Nothing crashed; the timings just quietly became meaningless, and
it was only caught by noticing the numbers looked odd. Now every run records the clock it
ran at, so that failure is visible *in the data* instead of being invisible.

Note it is sampled **outside** the timed region, so the probe itself can never affect a
measurement. And note what it does *not* threaten: tau is a count ratio, so clock speed
cannot touch it. Only wall-clock numbers are at risk.

In [ ]:
import subprocess

class PassCounter:
    def __init__(self, model):
        self.n = 0
        model.register_forward_hook(self._hook)
    def _hook(self, *_):
        self.n += 1

def gpu_clock_mhz():
    # Sampled OUTSIDE every timed region, so it cannot affect a measurement.
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=clocks.sm", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5).stdout.strip().splitlines()[0]
        return int(out)
    except Exception:
        return None

print("GPU clock right now:", gpu_clock_mhz(), "MHz")

## 5. Load the models onto the GPU
- `float16` = half precision, which halves memory so both fit in 8 GB
- `device_map="cuda"` puts them on the GPU (needs `accelerate`)
- VRAM check: if nearly full, Windows silently spills into normal RAM and the timings become garbage
- **Run this cell once.** If you need to reload, restart the kernel first

In [ ]:
assert torch.cuda.is_available(), "CUDA not available - fix the GPU first."
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU)

def load(name):
    m = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16, device_map="cuda")
    m.eval()
    return m

tok = AutoTokenizer.from_pretrained(TARGET)
target = load(TARGET)
draft = load(DRAFT)
counter = PassCounter(target)

used = torch.cuda.memory_allocated() / 2**30
total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"VRAM: {used:.1f} / {total:.1f} GB")
if used > 0.85 * total:
    print("WARNING: VRAM nearly full - switch TARGET to the 1.5B model.")

## 6. Pinning the draft strategy

In Transformers 4.57.6, default K=20 is constant with a confidence gate. With scikit-learn, confidence state can be written back across calls. Before every assisted call, pin K, the constant schedule and confidence threshold 0, disabling that adaptive behavior. K remains a maximum: EOS and the final generation budget may shorten a round.


In [ ]:
def set_draft_strategy(k):
    # Must run before EVERY assisted generate(): transformers mutates
    # assistant_confidence_threshold in place and the change persists on
    # draft.generation_config across calls.
    gc = draft.generation_config
    gc.num_assistant_tokens = int(k)
    gc.num_assistant_tokens_schedule = "constant"
    gc.assistant_confidence_threshold = 0.0   # disables early exit + ROC write-back

# What HF would have used if we never touched it:
print("HF defaults ->",
      "num_assistant_tokens =", draft.generation_config.num_assistant_tokens,
      "| schedule =", draft.generation_config.num_assistant_tokens_schedule,
      "| confidence_threshold =", draft.generation_config.assistant_confidence_threshold)
set_draft_strategy(4)
print("after pinning ->",
      "num_assistant_tokens =", draft.generation_config.num_assistant_tokens,
      "| confidence_threshold =", draft.generation_config.assistant_confidence_threshold)

## 7. Core functions

Encode the question with Qwen's chat template. Time synchronized target.generate, with assistant_model=draft for speculation. Set repetition_penalty explicitly; HF reuses target settings/processors for the assistant. do_sample=False selects the largest processed score, not necessarily the largest raw model logit.


In [ ]:
GREEDY = dict(do_sample=False, temperature=None, top_p=None, top_k=None)

def encode(prompt):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return tok(text, return_tensors="pt").to("cuda")

@torch.inference_mode()
def generate(inputs, max_new, k=0):
    # k=0 -> normal decoding. k>0 -> speculative with K=k draft tokens.
    kwargs = dict(max_new_tokens=max_new, pad_token_id=tok.eos_token_id,
                  repetition_penalty=REP_PENALTY, **GREEDY)
    if k:
        set_draft_strategy(k)                  # re-pin before every call
        kwargs["assistant_model"] = draft      # <-- speculative decoding switched on
    counter.n = 0
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = target.generate(**inputs, **kwargs)
    torch.cuda.synchronize(); dt = time.perf_counter() - t0
    new_ids = out[0, inputs["input_ids"].shape[1]:].tolist()
    return new_ids, dt, counter.n

## 8. Warm-up
- The first GPU run is always slow (loading kernels), so we do one throwaway run of each kind and don't count it

In [ ]:
warm = encode("Hello, how are you?")
generate(warm, 32, k=0)
generate(warm, 32, k=K_SWEEP[0])
print("warmed up")

## Calibration: generation-cost proxy

Measure synchronized wall time per generated token, including prefill and bookkeeping, with both models resident. Use three interleaved repeats. This is a cost proxy; a profiler is needed to identify overhead sources. `tau / (1 + K*c)` is an approximation, not a guaranteed bound for the measured implementation.


In [ ]:
from benchmark import calibrate_pass_cost
CALIBRATION_SAMPLES = {}
T_PASS, D_PASS, C = calibrate_pass_cost(target, draft, tok, counter,
                                     rep_penalty=REP_PENALTY, samples=CALIBRATION_SAMPLES)
print(f"target {T_PASS*1e3:.2f} ms/token proxy; draft {D_PASS*1e3:.2f}; c={C:.3f}")
print("Includes prefill/bookkeeping. Overhead sources were not profiled.")


## 10. Try ONE question and see it work
- Run this before the full sweep to understand what's happening
- Compare: time, passes, tau, and whether both answers are identical
- The speculative run makes **far fewer** big-model passes but can still take longer in
  wall-clock time. That gap is the whole story of this project

In [ ]:
q = encode("Explain how rain forms.")
a_ids, a_t, a_pass = generate(q, MAX_NEW, k=0)
b_ids, b_t, b_pass = generate(q, MAX_NEW, k=4)

print(f"normal:       {len(a_ids):3d} tokens in {a_t:.2f}s, {a_pass:3d} big-model passes, tau {len(a_ids)/a_pass:.2f}")
print(f"speculative:  {len(b_ids):3d} tokens in {b_t:.2f}s, {b_pass:3d} big-model passes, tau {len(b_ids)/max(b_pass,1):.2f}  (K=4)")
print(f"speedup: {(len(b_ids)/b_t)/(len(a_ids)/a_t):.2f}x | identical answers: {a_ids == b_ids}")
print("\nAnswer:\n", tok.decode(a_ids, skip_special_tokens=True))

## Shuffled sweep

Build baseline and speculative conditions for every question, shuffle each repeat, and record time, tokens and target calls. Shuffling spreads drift; it does not eliminate it. Medians of two trials equal their mean. Exact-match checks run after baselines are available. Saved IDs/text allow later inspection of divergences.


In [ ]:
prompts = json.loads((HERE / "prompts.json").read_text(encoding="utf-8"))

conditions = []
for lang in LANGS:
    for i in range(len(prompts[lang][:LIMIT])):
        conditions.append((lang, i, 0))          # 0 = baseline
        for k in K_SWEEP:
            conditions.append((lang, i, k))

rows, canonical, ids_by_row = [], {}, []
rng = random.Random(SEED)
total_runs = len(conditions) * REPEATS
done = 0
t_start = time.perf_counter()

for rep in range(REPEATS):
    order = conditions[:]
    rng.shuffle(order)                            # spread thermal drift across conditions
    for lang, i, k in order:
        inputs = encode(prompts[lang][i])
        ids, dt, passes = generate(inputs, MAX_NEW, k=k)
        key = (lang, i)
        if k == 0 and key not in canonical:
            canonical[key] = ids
        done += 1
        tau = len(ids) / max(passes, 1)
        ids_by_row.append(ids)
        rows.append(dict(
            repeat=rep, lang=lang, prompt_id=i, k=k,
            mode=("baseline" if k == 0 else f"spec_k{k}"),
            tokens=len(ids), sec=dt, tps=len(ids)/dt,
            passes=passes, tau=tau,
            matches_baseline=None,        # resolved after the loop, see below
            sm_mhz=gpu_clock_mhz(),       # sampled outside the timed region
            wall=round(time.perf_counter() - t_start, 1),
        ))
        label = "baseline" if k == 0 else f"K={k}"
        print(f"[{done:3d}/{total_runs}] {lang} p{i} {label:>9s} | "
              f"{len(ids)/dt:6.1f} tok/s | {passes:4d} passes | tau {tau:.2f}")

# Resolve the identical-output check only now. The order is shuffled, so a
# speculative run can land before that question's first baseline; comparing
# inside the loop would score those as "not identical".
for r, ids in zip(rows, ids_by_row):
    r["matches_baseline"] = ids == canonical[(r["lang"], r["prompt_id"])]

print(f"\ndone in {(time.perf_counter()-t_start)/60:.1f} min")
print("all outputs identical to baseline:", all(r["matches_baseline"] for r in rows))

## Output evidence and script screen

Keep all baseline and speculative IDs/text, exact match flags and first divergent token positions. Screen every output independently. Whitespace segments and Unicode code points are descriptive text measures, not comparable units of meaning.


In [ ]:
from benchmark import finalise_rows, text_metrics

evidence = finalise_rows(rows, ids_by_row, canonical, tok)
(OUT / "run_outputs.json").write_text(json.dumps(evidence, ensure_ascii=False, indent=2), encoding="utf-8")
samples = []
for (lang, i), ids in canonical.items():
    answer, meta = text_metrics(tok, ids, lang)
    samples.append(dict(lang=lang, prompt_id=i, prompt=prompts[lang][i], output=answer, **meta))
print("Exact speculative matches:", sum(r["matches_baseline"] for r in rows if r["k"]),
      "/", sum(bool(r["k"]) for r in rows))


## Summary

Screen every actual output by Unicode script; this does not assess language quality. Use median throughput within a prompt and then median prompt speedup ratios. Tau includes the target bonus/correction token. With two repeats the median equals the mean. SD across prompt ratios is variation across questions, not a confidence interval.


In [ ]:
from types import SimpleNamespace
from benchmark import summarise as aggregate, report

ARGS = SimpleNamespace(target=TARGET, draft=DRAFT, max_new=MAX_NEW,
                       repeats=REPEATS, seed=SEED, k=K_SWEEP,
                       rep_penalty=REP_PENALTY, langs=LANGS)
summary = aggregate(rows, ARGS, GPU, T_PASS, D_PASS, C)
report(summary, ARGS)


## Charts

Show measured median speedup and pooled tokens per target pass. The latter includes correction/bonus tokens. Error bars on the best-K bars show SD across question ratios, not a confidence interval. Compare only tested K values.


In [ ]:
from benchmark import plot
import matplotlib.pyplot as plt

plot(summary, ARGS, OUT)
for filename in ("sweep.png", "speedup.png"):
    plt.figure(figsize=(10, 4))
    plt.imshow(plt.imread(OUT / filename)); plt.axis("off"); plt.show()


## 15. Save everything
- `raw.csv`: every single trial, with its repeat number and wall-clock time
- `summary.json`: the per-language, per-K table plus `c`
- `outputs.json`: the actual answers; **open this and eyeball the Hindi/Urdu ones**
  (expect short, partly code-switched, sometimes plain wrong answers — report that honestly)

In [ ]:
with open(OUT / "raw.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
(OUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(OUT / "outputs.json").write_text(json.dumps(samples, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved to", OUT)
import platform, transformers
manifest = dict(target=TARGET, draft=DRAFT, gpu=GPU, max_new=MAX_NEW,
                repeats=REPEATS, k_sweep=K_SWEEP, seed=SEED, rep_penalty=REP_PENALTY,
                python=platform.python_version(), torch=torch.__version__,
                transformers=transformers.__version__, cuda=torch.version.cuda,
                target_device=str(target.device), draft_device=str(draft.device),
                target_revision=getattr(target.config, "_commit_hash", None),
                draft_revision=getattr(draft.config, "_commit_hash", None),
                calibration_seconds_per_token=CALIBRATION_SAMPLES,
                calibration="three interleaved end-to-end measurements; includes prefill",
                target_generation_config=target.generation_config.to_dict(),
                draft_generation_config=draft.generation_config.to_dict())
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
